<a href="https://colab.research.google.com/github/liuxiaohu0511/lance-demo/blob/develop/lance_performance_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

以下是 **Lance 性能指南（Lance Performance Guide）** 的中文精要翻译与解读，帮助你快速理解各部分重点👇

---

## 🚀 Lance 性能优化指南（概要）

本指南提供优化 Lance 应用性能的技巧与建议。

---

### 🪵 日志（Logging）

Lance 使用 Rust 的 `log` crate 记录日志，输出方式取决于客户端语言：

* **Rust**：需要手动配置日志订阅器（subscriber）。
* **Python / Java**：默认配置输出到 `stderr`。

#### 🔧 环境变量配置：

| 环境变量                     | 功能                         | 可选值 / 示例                         |
| ------------------------ | -------------------------- | -------------------------------- |
| `LANCE_LOG`              | 控制日志级别与过滤目标，取代 `RUST_LOG`。 | 例如：`info,lance::io_events=debug` |
| `LANCE_LOG_STYLE`        | 是否启用日志颜色。                  | `auto`, `always`, `never`        |
| `LANCE_LOG_TS_PRECISION` | 日志时间戳精度。                   | `ns`, `us`, `ms`, `s`            |
| `LANCE_LOG_FILE`         | 将日志重定向到文件，而不是标准错误输出。       | 例如：`/tmp/lance.log`              |

> ⚠️ 如果文件无法创建（例如权限不足），将自动退回 stderr。

---

### 🧭 Trace 事件（Trace Events）

Lance 使用 **tracing** 框架记录详细事件。

* Python 端（pylance）会将 trace 事件以日志消息输出。
* Rust 端可使用 `tracing` crate 进行捕获和分析。

---

### 📂 文件审计（File Audit）

当 Lance 创建或删除重要文件时，会触发 file audit 事件。

| 事件                  | 参数     | 描述                                                           |
| ------------------- | ------ | ------------------------------------------------------------ |
| `lance::file_audit` | `mode` | I/O 操作模式（`create`, `delete`, `delete_unverified`）            |
| `lance::file_audit` | `type` | 文件类型（`manifest`, `data file`, `index file`, `deletion file`） |

---

### 💾 I/O 事件（I/O Events）

表示重要的 I/O 操作（主要用于索引）。
这些事件不会在缓存命中时触发，可帮助调试缓存利用情况。

| 事件                 | 参数     | 描述                                                                    |
| ------------------ | ------ | --------------------------------------------------------------------- |
| `lance::io_events` | `type` | 操作类型，如 `open_scalar_index`, `open_vector_index`, `load_vector_part` 等 |

---

### ⚙️ 执行事件（Execution Events）

在执行计划运行时触发，用于分析查询性能。

| 参数                  | 描述                        |
| ------------------- | ------------------------- |
| `type`              | 当前执行事件类型（目前仅有 `plan_run`） |
| `output_rows`       | 输出行数                      |
| `iops`              | I/O 操作次数                  |
| `bytes_read`        | 读取的字节数                    |
| `indices_loaded`    | 加载的索引数量                   |
| `parts_loaded`      | 加载的索引分区数量                 |
| `index_comparisons` | 索引内执行的比较次数                |

---

### 🧵 线程模型（Threading Model）

Lance 是线程安全的。
多数操作会自动并行执行，内部有两个线程池：

| 线程池                     | 用途      | 默认线程数             | 环境变量                |
| ----------------------- | ------- | ----------------- | ------------------- |
| **IO Thread Pool**      | 负责磁盘读写  | 本地存储默认 8，云存储默认 64 | `LANCE_IO_THREADS`  |
| **Compute Thread Pool** | 数据计算与解码 | 与 CPU 核心数相同       | `LANCE_CPU_THREADS` |

> 🧠 建议：如果你在云环境中（如 S3），可适当将 IO 线程数增至 128 或 256，以充分利用带宽。

---

### 🧠 内存需求（Memory Requirements）

Lance 设计为流式（streaming）执行，不会整体加载数据集。
但以下组件可能占用大量内存：

---

#### 📘 元数据缓存（Metadata Cache）

* 存储表的 manifest、索引元数据、事务信息等。
* LRU 缓存（按字节计），默认大小：**1 GiB**。
* 每个表独立维护，不共享。
  → 建议复用同一个表或 Session 来共享缓存。

---

#### 📗 索引缓存（Index Cache）

* 缓存向量 / 标量索引到内存中，加速查询。
* 可通过 `index_cache_size_bytes` 参数设置（默认 **6 GiB**）。
* 自 v0.30.0 起，旧参数 `index_cache_size`（以 entries 为单位）已弃用。

---

### 🔍 扫描数据（Scanning Data）

扫描操作通常是流式执行，但需要一定缓冲区。

#### 💡内存估算：

内存约为：

> (2 × `io_buffer_size`) + (`batch_size` × `num_compute_threads`)

| 参数               | 默认值    | 说明                   |
| ---------------- | ------ | -------------------- |
| `io_buffer_size` | 2 GiB  | 用于磁盘页缓存（每页约 8–32 MB） |
| `batch_size`     | 8192 行 | 每次解码的行数（影响 CPU 内存使用） |

示例：

* 对于 1024 维向量（float32），8192 行 ≈ 32 MB；
* 若 16 CPU 线程并行，则需约 512 MB 计算内存；
* 可将 `batch_size` 降至 1024 控制内存使用。

> ⚠️ 过大的 `batch_size` 会导致内存占用激增；
> 过小则影响吞吐性能。

---

#### ⚙️ 全局 I/O 限制：

* 由 `LANCE_PROCESS_IO_THREADS_LIMIT` 控制（默认 128）。
* 设为 0 可取消限制，但可能导致存储系统重试或超时。

---

## ✅ 总结建议

1. 使用环境变量精细控制线程数与日志。
2. 尽量共享 Dataset 或 Session，避免重复缓存。
3. 对大数据扫描任务调整 `batch_size` 与 `io_buffer_size`。
4. 监控执行事件与 I/O 事件日志，调优瓶颈。
5. 云环境可增加 IO 线程至 128+ 提升带宽利用率。
